This document analyzes the dataset logged by CARLA's system. The metrics to be measured are the following: 
1. **Reaction Time**
    - This is from the time the alert was sent until the time the driver made an action. This will help the researches know the appropriate time to give an alert. 
2. **Completion Time**
    - This measures the duration of the whole violation. From the alert being triggered until the speeding gets resolved. 
3. **Ignored Alert** 
    - This checks whether the driver made an action to the alert or ignored it. This will be used to prove the effectiveness of providing an alert to decrease speed.
4. **Speed Change**
    - This checks the speed. 
5. **Alert Effectiveness**
    - This computes for the number of resolved speeding violations, where the rider either *reduced throttle* or *brakes* which is then divided by all violations. (resolved/all violations) 

*Note: An action made by the driver can be either a release of the throttle or pressing on the brakes.* 

In [249]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

In [250]:
df = pd.read_csv("cleaned_Sim.csv")

df.head()


,timestamp,phase,scenario,speed_kmh,event,details,speed_limit,overspeed,reduce_throttle,resolved,event_id
0,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,False,False,False,0
1,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,False,False,False,0
2,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,False,False,False,0
3,1.771924e+09,4,WEBSOCKET,1.17,WS_SEND,"{""phase"": 4, ""speed"": 1.17, ""speed_limit"": 30,...",30.0,False,False,False,0
4,1.771924e+09,4,WEBSOCKET,1.19,WS_SEND,"{""phase"": 4, ""speed"": 1.19, ""speed_limit"": 30,...",30.0,False,False,False,0


In [251]:
df.columns

Index(['timestamp', 'phase', 'scenario', 'speed_kmh', 'event', 'details',
       'speed_limit', 'overspeed', 'reduce_throttle', 'resolved', 'event_id'],
      dtype='object')

In [252]:
df["scenario"].unique()

array(['WEBSOCKET', 'TRAFFIC_LIGHT', 'SPEED_LIMIT', 'STOP', 'COLLISION',
       'HUD'], dtype=object)

In [253]:
df.loc[
    (df["event_id"]==5),
    ["speed_kmh", "speed_limit", "details", "event_id"]
]

,speed_kmh,speed_limit,details,event_id
7128,25.22,NaN,action=REDUCE_THROTTLE|time=2.23s|control={'th...,5
7129,48.31,NaN,action=NO_REACTION|time=3.02s|control={'thrott...,5
7130,48.77,30.0,Speed 48.77 km/h > limit 30.00 km/h at Locatio...,5
7131,36.93,NaN,action=VIOLATION_RESOLVED|time=5.16s|reaction_...,5


In [255]:
df.loc[
    df["event_id"].isin([183]),
    ["phase", "speed_kmh", "speed_limit", "details", "event_id"]
]

,phase,speed_kmh,speed_limit,details,event_id
135418,3,21.77,NaN,action=REDUCE_THROTTLE|time=0.91s|control={'th...,183
135419,3,21.77,30.0,"{""phase"": 3, ""speed"": 21.77, ""speed_limit"": 30...",183
135420,3,21.43,30.0,"{""phase"": 3, ""speed"": 21.43, ""speed_limit"": 30...",183
135421,3,20.95,30.0,"{""phase"": 3, ""speed"": 20.95, ""speed_limit"": 30...",183
135422,3,20.95,30.0,"{""phase"": 3, ""speed"": 20.95, ""speed_limit"": 30...",183
...,...,...,...,...,...
135758,3,39.22,30.0,"{""phase"": 3, ""speed"": 39.22, ""speed_limit"": 30...",183
135759,3,38.74,30.0,"{""phase"": 3, ""speed"": 38.74, ""speed_limit"": 30...",183
135760,3,37.15,30.0,"{""phase"": 3, ""speed"": 37.15, ""speed_limit"": 30...",183
135761,3,47.87,30.0,Speed 47.87 km/h > limit 30.00 km/h at Locatio...,183


In [256]:
df.loc[df["speed_limit"] == 60, "event_id"].unique()

array([  6,   0,  17,  18,  30,  52, 125, 139, 186, 244, 259, 279, 298,
       299, 300, 301, 310, 341, 348, 421, 431, 441, 505, 506, 533, 543])

## Metrics Computation & Analysis

In [257]:
df_Metrics = df.copy()

### Reaction Time

*action=REDUCE_THROTTLE* is a signal that the driver released yung tapak niya sa pedal. 

In [264]:
df.loc[
    df["event_id"].isin([7]),
    ["speed_kmh", "speed_limit", "details", "event_id"]
]

,speed_kmh,speed_limit,details,event_id
13107,37.29,30.0,"{""phase"": 3, ""speed"": 37.29, ""speed_limit"": 30...",7
13108,37.51,30.0,"{""phase"": 3, ""speed"": 37.51, ""speed_limit"": 30...",7
13109,37.88,30.0,"{""phase"": 3, ""speed"": 37.88, ""speed_limit"": 30...",7
13110,37.88,30.0,"{""phase"": 3, ""speed"": 37.88, ""speed_limit"": 30...",7
13111,38.18,30.0,"{""phase"": 3, ""speed"": 38.18, ""speed_limit"": 30...",7
...,...,...,...,...
13176,37.75,30.0,"{""phase"": 3, ""speed"": 37.75, ""speed_limit"": 30...",7
13177,37.75,30.0,"{""phase"": 3, ""speed"": 37.75, ""speed_limit"": 30...",7
13178,37.75,30.0,"{""phase"": 3, ""speed"": 37.75, ""speed_limit"": 30...",7
13179,48.73,30.0,Speed 48.73 km/h > limit 30.00 km/h at Locatio...,7


In [261]:
df_Metrics["carla_reaction_time"] = df_Metrics["details"].str.extract(
    r"action=REDUCE_THROTTLE\|time=(\d+\.\d+)s"
).astype(float)

In [262]:
carla_reactions = df_Metrics.loc[
    df_Metrics["carla_reaction_time"].notna(),
    ["event_id", "carla_reaction_time"]
]

In [265]:
display(carla_reactions.head(15))

,event_id,carla_reaction_time
767,1,0.76
5496,2,1.86
6252,3,0.76
7122,4,0.21
7128,5,2.23
7630,6,1.26
15595,8,0.66
15598,8,1.92
15601,9,0.18
15606,10,0.99


*Note: in the cases of NaN values: at times the rider is just above the speeding threshold, the system in CARLA might not have detected enough throttle release. Therefore, in the subtracting the action to the start becomes not applicable.*

### Completion Time

In [ ]:
# get timestamp and event_id
event_end_time = (
    df_Metrics[df_Metrics["details"].str.contains("VIOLATION_RESOLVED", na=False)]
    .groupby("event_id")["timestamp"]
    .first()
)

In [ ]:
completion_time = event_end_time - event_start_time

In [ ]:
display(completion_time.head(20))

event_id
0         NaN
1        1.30
2        2.26
3        1.29
4        0.05
5      297.08
6      256.25
7        4.04
8       60.96
9        7.42
10       0.19
11       0.03
12       0.54
13       0.57
14       0.71
15    1094.51
16     669.83
17    1421.70
18       2.12
19       4.44
Name: timestamp, dtype: float64

In [266]:
df_Metrics["carla_completion_time"] = df_Metrics["details"].str.extract(
    r"action=VIOLATION_RESOLVED\|time=(\d+\.\d+)s"
).astype(float)

In [267]:
carla_complete = df_Metrics.loc[
    df_Metrics["carla_completion_time"].notna(),
    ["event_id", "carla_completion_time"]
]

In [ ]:
display(carla_complete.head(30))

,event_id,carla_completion_time
780,1,1.33
5506,2,2.30
6265,3,1.32
7124,4,0.26
7131,5,5.16
11708,6,0.26
13180,7,4.07
15600,8,2.60
15603,8,0.45
15609,9,0.99


### Ignored Alert

In [268]:
ignored = (carla_reactions >= 2.5).astype(int)

In [ ]:
reaction_table["ignored"] = (
    (reaction_table["reaction_time"] >= 2.5)
).astype(int)

In [ ]:
display(reaction_table.tail(15))

,event_id,reaction_time,ignored
300,374,1.771679e+09,1
301,375,1.771679e+09,1
302,376,1.771679e+09,1
303,377,1.771679e+09,1
304,378,1.771679e+09,1
305,379,1.771679e+09,1
306,380,1.771679e+09,1
307,382,1.771681e+09,1
308,383,1.771681e+09,1
309,385,1.771681e+09,1


### Speed Change

### Alert Effectiveness

In [ ]:
speedingCount = df_Metrics["details"].str.contains("VIOLATION_RESOLVED", na=False).sum()
total_events = df_Metrics["event_id"].max()

In [ ]:
print("Speeding Count: ", speedingCount)
print("Total events: ", total_events)

Speeding Count:  611
Total events:  480


In [ ]:
effective = speedingCount/total_events

In [ ]:
print(effective)

1.2729166666666667


## Alert-Phase Analysis

In [ ]:
# Transfer to Sim Analysis

#Baseline
df_NC = df[df['phase'] == 1]
display(df_NC)

# Audio Only 
df_AO = df[df['phase'] == 2]
#display(df_AO)

# Haptic Only
df_HO = df[df['phase'] == 3]
#display(df_HO)

# Combination Haptic and Audio
df_HA = df[df['phase'] == 4]
#display(df_HA)


,timestamp,phase,scenario,speed_kmh,event,details,speed_limit,violation_start,violation_end,violation_state,event_start,event_id,overspeed_start,resolved,reaction
7122,1.771924e+09,1,SPEED_LIMIT,37.14,REACTION,action=REDUCE_THROTTLE|time=0.21s|control={'th...,NaN,False,False,1,1,4,False,False,True
7123,1.771924e+09,1,SPEED_LIMIT,37.14,SPEED_VIOLATION,Speed 37.14 km/h > limit 30.00 km/h at Locatio...,30.0,False,False,1,0,4,False,False,False
7124,1.771924e+09,1,SPEED_LIMIT,36.53,REACTION,action=VIOLATION_RESOLVED|time=0.26s|reaction_...,NaN,False,True,1,0,4,False,True,False
7125,1.771924e+09,1,TRAFFIC_LIGHT,14.79,YELLOW_LIGHT_PASS,"Passed yellow at Location(x=-85.075447, y=113....",NaN,False,False,0,0,0,False,False,False
7126,1.771924e+09,1,TRAFFIC_LIGHT,19.89,RED_LIGHT_VIOLATION,TrafficLight Violation at Location(x=-22.61465...,NaN,False,False,0,0,0,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
437869,1.771680e+09,1,SPEED_LIMIT,36.00,REACTION,action=VIOLATION_RESOLVED|time=1.78s|reaction_...,NaN,False,True,1,0,471,False,True,False
437870,1.771680e+09,1,SPEED_LIMIT,37.01,SPEED_VIOLATION,Speed 37.01 km/h > limit 30.00 km/h at Locatio...,30.0,False,False,0,0,0,False,False,False
437871,1.771680e+09,1,SPEED_LIMIT,36.70,REACTION,action=VIOLATION_RESOLVED|time=0.15s|reaction_...,NaN,False,True,0,0,0,False,True,False
437872,1.771680e+09,1,STOP,475.51,PHASE_STOP,Baseline,NaN,False,False,0,0,0,False,False,False


In [ ]:
#speeding_count = df["overspeed"].sum()
#VIOLATION COUNTS
speeding_count = df[df["event"] == "SPEED_VIOLATION"].shape[0]
stoplight_countRed = df[df["event"] == "RED_LIGHT_VIOLATION"].shape[0]
stoplight_countYellow = df[df["event"] == "YELLOW_LIGHT_PASS"].shape[0]
print("Speed count: ", speeding_count)
print("Red Light count: ", stoplight_countRed)
print("Yellow Light count: ", stoplight_countYellow)

Speed count:  926
Red Light count:  44033
Yellow Light count:  91


In [ ]:
resolved_rows = df_Metrics[df_Metrics["details"].str.contains("VIOLATION_RESOLVED", na=False)]
resolved_rows.head(10)

,timestamp,phase,scenario,speed_kmh,event,details,speed_limit,violation_start,violation_end,violation_state,event_start,event_id,overspeed_start,resolved,reaction,slow_down,carla_reaction_time,carla_completion_time
780,1.771924e+09,4,SPEED_LIMIT,36.30,REACTION,action=VIOLATION_RESOLVED|time=1.33s|reaction_...,NaN,False,True,1,0,1,False,True,False,False,NaN,1.33
5506,1.771924e+09,4,SPEED_LIMIT,36.39,REACTION,action=VIOLATION_RESOLVED|time=2.30s|reaction_...,NaN,False,True,1,0,2,False,True,False,False,NaN,2.30
6265,1.771924e+09,4,SPEED_LIMIT,35.52,REACTION,action=VIOLATION_RESOLVED|time=1.32s|reaction_...,NaN,False,True,1,0,3,False,True,False,False,NaN,1.32
7124,1.771924e+09,1,SPEED_LIMIT,36.53,REACTION,action=VIOLATION_RESOLVED|time=0.26s|reaction_...,NaN,False,True,1,0,4,False,True,False,False,NaN,0.26
7131,1.771925e+09,1,SPEED_LIMIT,36.93,REACTION,action=VIOLATION_RESOLVED|time=5.16s|reaction_...,NaN,False,True,1,0,5,False,True,False,False,NaN,5.16
11708,1.771925e+09,3,SPEED_LIMIT,66.34,REACTION,action=VIOLATION_RESOLVED|time=0.26s|reaction_...,NaN,False,True,1,0,6,False,True,False,False,NaN,0.26
13180,1.771925e+09,3,SPEED_LIMIT,35.19,REACTION,action=VIOLATION_RESOLVED|time=4.07s|reaction_...,NaN,False,True,1,0,7,False,True,False,False,NaN,4.07
15600,1.771926e+09,2,SPEED_LIMIT,36.85,REACTION,action=VIOLATION_RESOLVED|time=2.60s|reaction_...,NaN,False,True,1,0,8,False,True,False,False,NaN,2.60
15603,1.771926e+09,2,SPEED_LIMIT,96.73,REACTION,action=VIOLATION_RESOLVED|time=0.45s|reaction_...,NaN,False,True,1,0,8,False,True,False,False,NaN,0.45
15609,1.771926e+09,2,SPEED_LIMIT,36.87,REACTION,action=VIOLATION_RESOLVED|time=0.99s|reaction_...,NaN,False,True,1,0,9,False,True,False,False,NaN,0.99


In [ ]:
df.loc[
    df["details"].str.contains("VIOLATION_RESOLVED", na=False),
    ["scenario", "speed_kmh", "speed_limit", "event", "details", "event_id"]
].head(20)

,scenario,speed_kmh,speed_limit,event,details,event_id
780,SPEED_LIMIT,36.30,NaN,REACTION,action=VIOLATION_RESOLVED|time=1.33s|reaction_...,1
5506,SPEED_LIMIT,36.39,NaN,REACTION,action=VIOLATION_RESOLVED|time=2.30s|reaction_...,2
6265,SPEED_LIMIT,35.52,NaN,REACTION,action=VIOLATION_RESOLVED|time=1.32s|reaction_...,3
7124,SPEED_LIMIT,36.53,NaN,REACTION,action=VIOLATION_RESOLVED|time=0.26s|reaction_...,4
7131,SPEED_LIMIT,36.93,NaN,REACTION,action=VIOLATION_RESOLVED|time=5.16s|reaction_...,5
11708,SPEED_LIMIT,66.34,NaN,REACTION,action=VIOLATION_RESOLVED|time=0.26s|reaction_...,6
13180,SPEED_LIMIT,35.19,NaN,REACTION,action=VIOLATION_RESOLVED|time=4.07s|reaction_...,7
15600,SPEED_LIMIT,36.85,NaN,REACTION,action=VIOLATION_RESOLVED|time=2.60s|reaction_...,8
15603,SPEED_LIMIT,96.73,NaN,REACTION,action=VIOLATION_RESOLVED|time=0.45s|reaction_...,8
15609,SPEED_LIMIT,36.87,NaN,REACTION,action=VIOLATION_RESOLVED|time=0.99s|reaction_...,9
